## FLANG masked-idiom probe

Does financial pretraining teach central bank register? FLANG adapts BERT and
RoBERTa to financial text, so the probe of \citet{gambacorta2024} answers this
directly. Each model is compared against the checkpoint it was built from.

Inference only, no training, so this runs on CPU in a couple of minutes.
Writes `idioms_flang.csv`.


### Colab Setup

In [1]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    # mounting is optional: without it, results land in the Colab session
    # filesystem and are lost when the runtime ends
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        print(f"Drive not mounted. Results will be written to {ROOT} "
              "and lost when the session ends.")

Mounted at /content/drive


### Key Imports

In [2]:
import pandas as pd
import torch

from config import IDIOMS, RESULTS_DIR
from transformers import pipeline

OUT = RESULTS_DIR / "idioms_flang.csv"
DEVICE = 0 if torch.cuda.is_available() else -1

# each FLANG model beside the checkpoint it was adapted from. BERT is uncased and
# RoBERTa cased, so only the within-family deltas are comparable.
MODELS = {
    "bert-base": "bert-base-uncased",
    "flang-bert": "SALT-NLP/FLANG-BERT",
    "roberta-base": "roberta-base",
    "flang-roberta": "SALT-NLP/FLANG-RoBERTa",
}
print("idioms:", len(IDIOMS), "| results ->", OUT)


idioms: 100 | results -> /content/drive/MyDrive/thesis/idioms_flang.csv


### Probe

Same rule as the adaptation arms. An item counts as solved if the held-out word
is among the five highest-probability predictions.


In [3]:
def probe_idioms(model_path):
    mlm = pipeline("fill-mask", model=model_path, device=DEVICE)
    mask = mlm.tokenizer.mask_token
    rows = []
    for phrase, gold in IDIOMS:
        top5 = [
            r["token_str"].strip().lower()
            for r in mlm(phrase.replace("[MASK]", mask), top_k=5)
        ]
        rows.append(dict(phrase=phrase, gold=gold, hit=gold.lower() in top5, top5="|".join(top5)))
    del mlm
    if DEVICE == 0:
        torch.cuda.empty_cache()
    return rows


records = []
for label, path in MODELS.items():
    rows = probe_idioms(path)
    records += [dict(model=label, **r) for r in rows]
    print(f"{label:16s} {sum(r['hit'] for r in rows)}/{len(rows)}", flush=True)

pd.DataFrame(records).to_csv(OUT, index=False)
print("saved ->", OUT)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/311 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/843k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


flang-roberta    67/100
saved -> /content/drive/MyDrive/thesis/idioms_flang.csv


### Within-family deltas

Financial pretraining against the checkpoint it started from.


In [4]:
d = pd.read_csv(OUT).groupby("model")["hit"].sum()
for base, flang in [("bert-base", "flang-bert"), ("roberta-base", "flang-roberta")]:
    print(f"{base:14s} {d[base]:3d}  ->  {flang:14s} {d[flang]:3d}   {d[flang] - d[base]:+d}")


bert-base       53  ->  flang-bert      72   +19
roberta-base    60  ->  flang-roberta   67   +7
